# Modul 2: Der Agent Loop — Denken, Handeln, Reflektieren

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du einen minimalen ReAct-Loop von Grund auf.

🎯 **Lernziel:** Den ReAct-Loop (Reason → Act → Observe) implementieren UND das Reflection-Pattern anwenden.

---

## Das ReAct-Pattern (Yao et al., ICLR 2023)

```
┌─────────┐     ┌─────────┐     ┌─────────┐
│  THINK  │ ──► │   ACT   │ ──► │ OBSERVE │
│(Reason) │     │ (Tool)  │     │(Result) │
└────▲────┘     └─────────┘     └────┬────┘
     │                               │
     └───────────────────────────────┘
              Feedback Loop
```

Die Autoren zeigen: **Reasoning + Acting zusammen verbessert die Task-Completion um 20-30%** gegenüber reinem Chain-of-Thought.

In [ ]:
# Ein minimaler ReAct-Agent (ohne LLM, nur die Struktur)

import random

def think(observation, goal):
    """Reasoning-Schritt: Was soll ich als nächstes tun?"""
    if observation is None:
        return f'Ich muss herausfinden: {goal}. Zuerst suche ich.'
    elif 'error' in str(observation).lower():
        return 'Das hat nicht funktioniert. Ich probiere einen anderen Ansatz.'
    else:
        return f'Ich habe Informationen: {observation}. Lass mich das verarbeiten.'

def act(thought):
    """Action-Schritt: Tool aufrufen"""
    if 'suche' in thought.lower():
        return {'tool': 'web_search', 'result': 'Python wurde 1991 von Guido van Rossum erstellt.'}
    elif 'verarbeiten' in thought.lower():
        return {'tool': 'summarize', 'result': 'Python: 1991, Guido van Rossum.'}
    else:
        return {'tool': 'retry', 'result': 'Neuer Versuch...'}

def observe(action_result):
    """Observation-Schritt: Was ist das Ergebnis?"""
    return action_result['result']

# ReAct-Loop ausführen
goal = 'Wann wurde Python erfunden?'
obs = None

for cycle in range(3):
    print(f'\n--- Zyklus {cycle + 1} ---')
    thought = think(obs, goal)
    print(f'💭 THINK: {thought}')
    
    result = act(thought)
    print(f'⚡ ACT:   [{result["tool"]}]')
    
    obs = observe(result)
    print(f'👁️ OBSERVE: {obs}')

## Reflection-Pattern (Shinn et al., 2023)

Der Agent **evaluiert sein eigenes Ergebnis** und iteriert:

```
Agent Output → Self-Evaluation → "Ist das gut genug?" 
                                    ├─ Ja → Antwort
                                    └─ Nein → Überarbeiten
```

Shinn et al. zeigen: Self-Evaluation über mehrere Iterationen **steigert die Performance um ~20%** auf Coding-Tasks.

In [ ]:
# 🎯 ÜBUNG: Füge Reflection zum ReAct-Loop hinzu

def reflect(observation, goal):
    """Reflection-Schritt: Ist mein Ergebnis gut genug?"""
    # TODO: Implementiere eine einfache Qualitätsprüfung
    # Tipp: Prüfe ob die Observation das Ziel beantwortet
    # Return: True (gut genug) oder False (nochmal)
    pass  # ← Ersetze das hier

# Test
print(reflect('Python: 1991, Guido van Rossum.', 'Wann wurde Python erfunden?'))
# Sollte True zurückgeben

In [ ]:
# ✅ LÖSUNG

def reflect(observation, goal):
    """Reflection: Prüfe ob das Ergebnis das Ziel beantwortet."""
    # Einfache Heuristik: Enthält die Antwort Schlüsselwörter aus der Frage?
    keywords = [w.lower() for w in goal.split() if len(w) > 3]
    obs_lower = observation.lower()
    matches = sum(1 for kw in keywords if kw in obs_lower)
    quality = matches / max(len(keywords), 1)
    
    print(f'  🪞 REFLECT: Qualität {quality:.0%} ({matches}/{len(keywords)} Keywords gefunden)')
    return quality >= 0.3  # Mindestens 30% der Keywords

# Test
print('Test 1:', reflect('Python: 1991, Guido van Rossum.', 'Wann wurde Python erfunden?'))
print('Test 2:', reflect('Das Wetter ist schön.', 'Wann wurde Python erfunden?'))

## Vergleich: ReAct in verschiedenen Frameworks

| Framework | ReAct-Implementierung | Loop-Kontrolle |
|-----------|----------------------|----------------|
| **LangChain** | AgentExecutor | max_iterations |
| **CrewAI** | Task Loop | max_iter per Task |
| **AutoGen** | ConversableAgent | max_consecutive_auto_reply |
| **OpenClaw** | Session Loop | Tool Policies + Approval |

---

### ✅ Self-Check
- [ ] Ich kann den ReAct-Loop (Think → Act → Observe) erklären
- [ ] Ich verstehe warum Reflection die Qualität verbessert
- [ ] Ich kann 3 Zyklen in Agent-Logs identifizieren

**→ Weiter zu [Modul 3: Architektur eines Orchestrators](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul3_architektur.ipynb)**